# Component 2 — Adaptive GEV Exceedance Probabilities

C2 reads IFS ensemble forecasts from the IceChunk store, computes rolling
precipitation accumulations over multiple windows, compares against GEV-fitted
CMORPH thresholds stratified by ENSO/IOD phase, and writes exceedance probabilities
to a Zarr store.

**Key modules**: `accumulations`, `exceedance`, `thresholds`

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from datetime import date

from gik_icechain.shared.config import load_config

# Load .env credentials
_env = Path("../.env")
if _env.exists():
    for _line in _env.read_text().splitlines():
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("AWS_ACCESS_KEY_ID", os.environ.get("MINIO_ACCESS_KEY", ""))
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", os.environ.get("MINIO_SECRET_KEY", ""))

cfg = load_config(Path("../configs/default.yaml"))
STORAGE_OPTIONS = {"endpoint_url": cfg.outputs.endpoint_url}

# Demo window — data available in the store
START = date(2025, 1, 1)
END   = date(2025, 1, 2)
print(f"Config loaded — endpoint: {cfg.outputs.endpoint_url}")
print(f"Demo window: {START} → {END}")

## 2.1 Load forecast from IceChunk

In [ ]:
def _synthetic_day_ds():
    """Synthetic mm-scale ensemble tp for offline/CI demo (no live store)."""
    rng = np.random.default_rng(0)
    nm, ns, ny, nx = 51, 8, 30, 30
    tp = np.cumsum(rng.exponential(8.0, (nm, ns, ny, nx)), axis=1).astype("float32")
    return xr.Dataset(
        {"tp": (["member", "step", "latitude", "longitude"], tp)},
        coords={"member": np.arange(nm), "step": np.arange(0, ns * 6, 6),
                "latitude": np.linspace(23, -13, ny), "longitude": np.linspace(22, 53, nx)},
    )

from gik_icechain.conversion.icechunk_writer import IceChainStore
LIVE_STORE = bool(cfg.outputs.endpoint_url)
day_ds = None
if LIVE_STORE:
    try:
        store = IceChainStore(
            cfg.outputs.icechunk_store_uri,
            region=cfg.outputs.icechunk_store_region,
            endpoint_url=cfg.outputs.endpoint_url,
        )
        store.create_or_open()
        session = store.readonly_session()
        day_ds = xr.open_zarr(session.store, group="2025-01-01", consolidated=False)[["tp"]]
        b = cfg.component2.spatial.bbox  # [lat_min, lon_min, lat_max, lon_max]
        day_ds = day_ds.sel(latitude=slice(b[2], b[0]), longitude=slice(b[1], b[3]))
    except Exception as exc:
        print(f"Store error ({type(exc).__name__}) -- synthetic demo data")
        day_ds = None
if day_ds is None:
    print("Offline: synthetic mm-scale demo data")
    day_ds = _synthetic_day_ds()

print(day_ds)

## 2.2 Rolling accumulations

In [ ]:
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations

WINDOWS_H = [24, 48, 72]
acc = compute_rolling_accumulations(day_ds, windows_h=WINDOWS_H)
for w in WINDOWS_H:
    key  = f"tp_{w}h"
    vals = acc[key]
    print(f"{key}: shape={vals.shape}  max={float(vals.max()):.4f} m")

## 2.3 Adaptive GEV thresholds

In [ ]:
from gik_icechain.exceedance.thresholds import (
    AdaptiveGEVThresholds, ClimateMode,
    classify_enso, classify_iod, get_season,
)

thresholds = AdaptiveGEVThresholds.load(Path("../data/cmorph_thresholds/"))

enso_iod = pd.read_csv("../data/enso_iod_index.csv", parse_dates=["date"]).set_index("date")
row    = enso_iod.loc[pd.Timestamp(START)]
enso   = classify_enso(float(row["nino34_anom"]))
iod    = classify_iod(float(row["dmi"]))
season = get_season(START.month)
mode   = ClimateMode(season, enso, iod)
print(f"Climate mode: {mode.key}")

thr_24h_5y = thresholds.get(24, 5, mode)
print(f"Threshold 24h/5yr: min={float(thr_24h_5y.min()):.3f} max={float(thr_24h_5y.max()):.3f} m")

## 2.4 Exceedance probabilities

In [ ]:
from gik_icechain.exceedance.exceedance import (
    compute_exceedance_probabilities, compute_ensemble_confidence,
)

p    = compute_exceedance_probabilities(acc, xr.Dataset({"rp_5y": thr_24h_5y}), 24, 5, "member")
conf = compute_ensemble_confidence(acc, window_h=24, member_dim="member")

print(f"Exceedance: min={float(p.min()):.3f} max={float(p.max()):.3f} mean={float(p.mean()):.3f}")
print(f"Confidence states: {set(int(v) for v in np.unique(conf.values))}")

## 2.5 Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
p.plot(ax=axes[0], cmap="YlOrRd", vmin=0, vmax=1)
axes[0].set_title("Exceedance prob — 24h/5yr")
conf.plot(ax=axes[1], cmap="RdYlGn", vmin=0, vmax=2)
axes[1].set_title("Ensemble confidence")
fig.tight_layout()
plt.show()

## 2.6 Exceedance Zarr store

In [ ]:
try:
    exc_ds = xr.open_zarr(
        cfg.outputs.exceedance_store_uri,
        consolidated=False,
        storage_options=STORAGE_OPTIONS,
    )
    print("Exceedance store dimensions:")
    print(exc_ds)
    dates = exc_ds.coords.get("date", exc_ds.coords.get("time", None))
    if dates is not None:
        print(f"Date range: {str(dates.values[0])[:10]} → {str(dates.values[-1])[:10]}")
except Exception as exc:
    print("C2 store not yet available — run gik-icechain exceedance first")
    print(f"({type(exc).__name__}: {exc})")